In [1]:
# -*- coding: utf-8 -*-
import os.path

import torch
from torch.utils.data import DataLoader

from scripts.llms import PretrainedGPT
from scripts.data_ops import TextDataset
from scripts.train_ops import training_loop_with_batch

from torch import optim

from torchinfo import summary
import json

In [2]:
# Set file name
file_names = "works_of_fela.txt"
tokenizer_name = os.path.splitext(file_names)[0] + ".json"

In [3]:
# Read hyperparameters from config files
model_kwargs = dict(json.load(open("../config/model_config.json")))
hyperparameter_kwargs = dict(
    json.load(open("../config/hyperparameters_config.json"))
)

In [4]:
# Model architecture hyperparameters

narrow = model_kwargs["narrow"]
num_heads = model_kwargs["num_heads"]
num_decoder_blocks = model_kwargs["num_decoder_blocks"]
num_embeddings = model_kwargs["num_embeddings"]
embedding_dim = model_kwargs["embedding_dim"]
max_token = model_kwargs["max_token"]
pre_ln = model_kwargs["pre_ln"]
share_weights = model_kwargs["share_weights"]
dropout_rate = model_kwargs["dropout_rate"]

In [5]:
# Data and training process hyperparameters

seq_len = hyperparameter_kwargs["seq_len"]
epochs = hyperparameter_kwargs["epochs"]
lr = hyperparameter_kwargs["lr"]
batch_size = hyperparameter_kwargs["batch_size"]
weight_decay = hyperparameter_kwargs["weight_decay"]
use_last = hyperparameter_kwargs["use_last"]

In [6]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
# Prepare datasets
train_dataset = TextDataset(
    file_name=file_names,
    tokenizer_path=tokenizer_name,
    time_steps=seq_len,
    min_length=seq_len,
    train=True,
    frac=0.85,
)

test_dataset = TextDataset(
    file_name=file_names,
    tokenizer_path=tokenizer_name,
    time_steps=seq_len,
    min_length=seq_len,
    train=False,
    frac=0.15,
)

Ignored unknown kwargs option inital_alphabet
Ignored unknown kwargs option inital_alphabet


In [8]:
# Generate dataloaders
train_dataloader = DataLoader(
    dataset=train_dataset, batch_size=batch_size, shuffle=True
)
test_dataloader = DataLoader(
    dataset=test_dataset, batch_size=batch_size, shuffle=False
)

In [9]:
# Sample data loader
x_sample, y_sample = next(iter(train_dataloader))
x_sample, y_sample = x_sample.to(device), y_sample.to(device)

In [10]:
print(x_sample)

tensor([[ 463,  271,  292,  ...,  132, 1406,   60],
        [ 296,   11,  296,  ...,   71,   11,  335],
        [  87,   87, 2066,  ...,   12,  135,  148],
        ...,
        [ 327,   92,   87,  ...,  402,  550,  132],
        [  60,  283,   95,  ...,   81,  348,  238],
        [ 296,   11,  418,  ...,   13,  461,   78]], device='cuda:0')


In [11]:
i = 0

for sample_row in x_sample:
    sample_row_ = sample_row.cpu().numpy().tolist()
    decoded_sample_row = train_dataset.data_processor.tokenizer.decode(sample_row_)
    print(i, ".", decoded_sample_row+"\n")
    i += 1

0 .  brother like that
 you tell am realize you be fool at twenty
 book sense different from belly sense
 fool na fool no matter for age
 fifteen to thirty, forty to hundred
 fool na fool no matter for age
 
 now listen to me now
 
 [chorus]
 yes sir!
 
 hey come
 bring me the plate
 go buy am to chop
 bring me the file
 call the messenger
 hey come
 bring me the plates
 go buy am to chop
 bring me the file
 call the messenger
 
 they run run about everywhere
 they jump jump about the place
 na carry for morning, day, and night
 
 i no sing happy monkey for you
 i no say i be fool for you
 
 [chorus]
 okay, i agree, i don vex me to hear!
 
 okay, i agree, i don vex me to hear
 okay, i agree, i don vex me to hear
 okay, i agree, i don vex me to hear
 okay, i agree, i don vex me to hear
 okay, i agree, i don vex me to hear
 
 before i jump like monkey, give me banana
 make you hear, yes sir
 
 give me banana
 give me banana


1 .  eh, eh
 ma s'oju mi o seri si mi o o
 je k'abere
 oya o o

In [12]:
i = 0

for sample_row in y_sample:
    sample_row_ = sample_row.cpu().numpy().tolist()
    decoded_sample_row = train_dataset.data_processor.tokenizer.decode(sample_row_)
    print(i, ".", decoded_sample_row+"\n")
    i += 1

0 .  like that
 you tell am realize you be fool at twenty
 book sense different from belly sense
 fool na fool no matter for age
 fifteen to thirty, forty to hundred
 fool na fool no matter for age
 
 now listen to me now
 
 [chorus]
 yes sir!
 
 hey come
 bring me the plate
 go buy am to chop
 bring me the file
 call the messenger
 hey come
 bring me the plates
 go buy am to chop
 bring me the file
 call the messenger
 
 they run run about everywhere
 they jump jump about the place
 na carry for morning, day, and night
 
 i no sing happy monkey for you
 i no say i be fool for you
 
 [chorus]
 okay, i agree, i don vex me to hear!
 
 okay, i agree, i don vex me to hear
 okay, i agree, i don vex me to hear
 okay, i agree, i don vex me to hear
 okay, i agree, i don vex me to hear
 okay, i agree, i don vex me to hear
 
 before i jump like monkey, give me banana
 make you hear, yes sir
 
 give me banana
 give me banana
 


1 . , eh
 ma s'oju mi o seri si mi o o
 je k'abere
 oya o o
 je k'ab

In [13]:
# Set TF32 precision
torch.set_float32_matmul_precision("high")

In [14]:

# Instantiate model
model = PretrainedGPT(
    narrow=narrow,
    num_heads=num_heads,
    num_decoder_blocks=num_decoder_blocks,
    num_embeddings=num_embeddings,
    embedding_dim=embedding_dim,
    max_token=max_token,
    pre_ln=pre_ln,
    share_weights=share_weights,
    dropout_rate=dropout_rate,
).to(device)


In [15]:
#
# # Compile model
# if hasattr(torch, "compile"):
#     print("Compiling model...")
#     model = torch.compile(model, backend="inductor")

In [16]:
# Instantiate optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=lr,
    weight_decay=weight_decay,
    betas=(.9, .95), # beta values for GPT-3
    foreach=True
)

In [ ]:
# Train model and return results
train_artefacts = training_loop_with_batch(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    scheduler=None,
    epochs=epochs,
    device=device,
    use_last=use_last,  # DO NOT CHANGE!
)

Epoch: 1 | Train loss:  47.499 | Test loss:  16.050 | Train perplexity:  425094909564989472768.000 | Test perplexity:  9344341.000 |  Train accuracy:  1.363 % | Test accuracy:  12.159 % |

Epoch: 2 | Train loss:  14.682 | Test loss:  9.225 | Train perplexity:  2379102.250 | Test perplexity:  10147.573 |  Train accuracy:  2.542 % | Test accuracy:  13.157 % |



In [ ]:
# Save config files for future reference
json.dump(model_kwargs, open(train_artefacts["model_config_path"], "w"))
json.dump(
    hyperparameter_kwargs, open(train_artefacts["hyperparameter_config_path"], "w")
)

In [ ]:
# Extract model instance as variable and set it to eval mode
model = train_artefacts["model"]
model.eval()

In [ ]:
# Freeze model weights
for param in train_artefacts["model"].parameters():
    param.requires_grad = False

In [ ]:
out = train_artefacts["model"](x_sample)

In [ ]:
summary(train_artefacts["model"], input_data=x_sample)

In [ ]:
print("Number of embeddings:", num_embeddings)
print("Final output shape:", out.shape)